# Method 4: Mirror-conditioned StyleGAN (supervised)
Same model and hyper-parameters as method 3, plus a horizontally flipped copy of the masked image and mask as input (8 channels). Ablation: 03 (baseline) vs 04a (mirror input) vs 04b (mirror input + symmetry loss). Expect the biggest gain on one-sided holes, little on centre holes.

**Code lives in** `src/methods/mirror_conditioned.py`. This notebook only drives it: train / run, evaluate, save results.

Status: method file is a stub; the cells below run once it is implemented.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/EricaCleo/DLpastiA.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm pandas
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(os.getcwd(), '| torch', torch.__version__, '| device:', dev)
if dev == 'cpu': print('NO GPU: on Colab use Runtime > Change runtime type > GPU')

## 1. Data

In [ ]:
# Data (run notebook 00 once first, or this downloads it). Needs env var KAGGLE_API_TOKEN.
from pathlib import Path
if not Path('data/raw').exists():
    !python scripts/download_data.py
from src.data import get_datasets, get_loaders, load_config
cfg = load_config()
train_loader, _ = get_loaders(batch_size=16)
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))

## 2. Train both variants

In [ ]:
from src.methods.mirror_conditioned import MirrorConditionedInpainter
variants = {'mirror_04a': MirrorConditionedInpainter(use_symmetry_loss=False, device=dev),
            'mirror_04b': MirrorConditionedInpainter(use_symmetry_loss=True, sym_weight=0.05, device=dev)}
for name, m in variants.items():
    m.name = name
    m.train(train_loader, epochs=100)

## 3. Evaluate the ablation

In [ ]:
from src.eval import evaluate, save_result
for name, m in variants.items():
    for mask in ['square', 'one_sided', 'centre']:
        res = evaluate(m, mask, n_images=200, device=dev)
        save_result(name, mask, res); print(name, mask, res)